In [ ]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

date = datetime.now().strftime("%Y-%m-%d")

verbose = 0
data_set = "gmtkn-cc-pVDZ"

data_path_list = sorted(
    list(Path("../validate").glob(f"*gmtkn*.csv")),
    key=lambda p: p.stat().st_ctime,
)

basis_args = "cc-pVDZ"
print(basis_args)

with open(f"new_dataset/{data_set}.json") as f:
    json_data = json.load(f)

with open(f"./subset.json") as f:
    full_subset_dict = json.load(f)["full_subset_dict"]
    # full_subset_dict = json.load(f)["full_subset_dict_test"]

data_subset = {}

for name_set, subset_list_ in full_subset_dict.items():
    for data_path in data_path_list:
        data = pd.read_csv(data_path)
        data_name = (data["name"].str.split(f"_{basis_args}").str[0]).to_numpy()
        data_dft = data["dft_ene"].to_numpy() * 627.5094733748099
        data_scf = data["scf_ene"].to_numpy() * 627.5094733748099
        data_cc = data["cc_ene"].to_numpy() * 627.5094733748099
        if "delta_d3bj" in data.columns:
            data_d3bj = data["delta_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_d3bj = np.zeros_like(data_dft)
        if "delta_d3zero" in data.columns:
            data_d3zero = data["delta_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_d3zero = np.zeros_like(data_dft)

        if modified_dft_d3bj := data.get("modified_dft_d3bj"):
            data_dft_d3bj = modified_dft_d3bj.to_numpy()
        else:
            data_dft_d3bj = data_d3bj
        if modified_dft_d3zero := data.get("modified_dft_d3zero"):
            data_dft_d3zero = modified_dft_d3zero.to_numpy()
        else:
            data_dft_d3zero = data_d3zero

        if modified_ai_d3bj := data.get("modified_ai_d3bj"):
            data_ai_d3bj = modified_ai_d3bj.to_numpy()
        else:
            data_ai_d3bj = data_d3bj
        if modified_ai_d3zero := data.get("modified_ai_d3zero"):
            data_ai_d3zero = modified_ai_d3zero.to_numpy()
        else:
            data_ai_d3zero = data_d3zero

        del data_d3bj, data_d3zero

        for i_subset in subset_list_:
            data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
            name_subset = f"{data_path_name}_{i_subset}"
            data_subset[name_subset] = {
                "name": [],
                "dft": [],
                "dft_d3bj": [],
                "dft_d3zero": [],
                "ai": [],
                "ai_d3bj": [],
                "ai_d3zero": [],
                "cc": [],
            }

            reaction_dict = json_data[f"reaction-{i_subset}"]
            for i_reaction_name, i_reaction in reaction_dict.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                atomic_energy_dft_d3bj = 0
                atomic_energy_dft_d3zero = 0
                atomic_energy_ai = 0
                atomic_energy_ai_d3bj = 0
                atomic_energy_ai_d3zero = 0
                atomic_energy_cc = 0
                finished, exist = True, True

                for i in range(len(systems_list)):
                    mole_name = (
                        systems_list[i]
                        if i_subset == "BH76RC"
                        else f"{i_subset}-{systems_list[i]}"
                    )
                    stoichiometry = int(stoichiometry_list[i])

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished, exist = False, False
                        if verbose > 0:
                            print(f"Warning: {mole_name} not found in data json, ERROR")
                        break

                    col = np.where(data_name == mole_name)[0]
                    if col.size == 1:
                        atomic_energy_dft += data_dft[col[0]] * stoichiometry
                        atomic_energy_dft_d3bj += data_dft_d3bj[col[0]] * stoichiometry
                        atomic_energy_dft_d3zero += (
                            data_dft_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_ai += data_scf[col[0]] * stoichiometry
                        atomic_energy_ai_d3bj += data_ai_d3bj[col[0]] * stoichiometry
                        atomic_energy_ai_d3zero += (
                            data_ai_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_cc += data_cc[col[0]] * stoichiometry
                    else:
                        finished = False
                        if verbose > 0:
                            print(
                                f"Warning: {mole_name} not found in {data_path.stem} data file"
                            )
                        break

                if exist:
                    data_subset[name_subset]["name"].append(i_reaction_name)
                if finished:
                    data_subset[name_subset]["dft"].append(
                        abs(atomic_energy_dft - atomic_energy_cc)
                    )
                    data_subset[name_subset]["dft_d3bj"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3bj
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["dft_d3zero"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["ai"].append(
                        abs(atomic_energy_ai - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3bj"].append(
                        abs(atomic_energy_ai + atomic_energy_ai_d3bj - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3zero"].append(
                        abs(
                            atomic_energy_ai
                            + atomic_energy_ai_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["cc"].append(abs(atomic_energy_cc))
                    if np.abs(atomic_energy_cc) > 1000:
                        print(
                            f"Warning: {i_reaction_name} in {name_subset} has a large CC energy: {atomic_energy_cc} kcal/mol"
                        )

            for key, val in data_subset.items():
                for key2, val2 in val.items():
                    if isinstance(val2, list):
                        data_subset[key][key2] = np.array(val2)

data_path_name_list = [
    data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    for data_path in data_path_list
]
header = pd.MultiIndex.from_product(
    [
        data_path_name_list,
        [
            "AI",
            "DFT",
            "AI_D3BJ",
            "DFT_D3BJ",
            "AI_D3ZERO",
            "DFT_D3ZERO",
            "Processed",
        ],
    ],
    names=["data_path", "Disp type"],
)

df_summary_subset = pd.DataFrame(columns=header)
mean_subset = pd.DataFrame(columns=header)
wtmad_1_subset = pd.DataFrame(columns=header)
wtmad_2_subset = pd.DataFrame(columns=header)
for data_path in data_path_list:
    mean_absolute_deviation_list = []
    data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    for name_set, subset_list_ in full_subset_dict.items():
        subset_ai = {}
        wtmad_1_ai = {}
        wtmad_2_ai = {}
        subset_dft = {}
        wtmad_1_dft = {}
        wtmad_2_dft = {}
        for d3_name in ["", "_d3bj", "_d3zero"]:
            subset_ai[d3_name] = []
            wtmad_1_ai[d3_name] = []
            wtmad_2_ai[d3_name] = []
            subset_dft[d3_name] = []
            wtmad_1_dft[d3_name] = []
            wtmad_2_dft[d3_name] = []
        processed = []
        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            if len(data_subset[name_subset]["ai"]) == 0:
                for col_name in [
                    "AI",
                    "DFT",
                    "AI_D3BJ",
                    "DFT_D3BJ",
                    "AI_D3ZERO",
                    "DFT_D3ZERO",
                ]:
                    df_summary_subset.loc[i_subset, (data_path_name, col_name)] = 0
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    f"0 / {len(data_subset[name_subset]['name'])}"
                )
            else:
                for d3_name in ["", "_d3bj", "_d3zero"]:
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"AI{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"ai{d3_name}"])
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"DFT{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"dft{d3_name}"])
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    f"{len(data_subset[name_subset]['ai'])} / "
                    f"{len(data_subset[name_subset]['name'])}"
                )

                if np.mean(data_subset[name_subset]["cc"]) > 75:
                    wtmad_1 = 0.1
                elif np.mean(data_subset[name_subset]["cc"]) < 7.5:
                    wtmad_1 = 10
                else:
                    wtmad_1 = 1

                for d3_name in ["", "_d3bj", "_d3zero"]:
                    subset_ai[d3_name] = np.append(
                        subset_ai[d3_name], data_subset[name_subset][f"ai{d3_name}"]
                    )
                    subset_dft[d3_name] = np.append(
                        subset_dft[d3_name], data_subset[name_subset][f"dft{d3_name}"]
                    )
                    wtmad_1_ai[d3_name] = np.append(
                        wtmad_1_ai[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"ai{d3_name}"]),
                    )
                    wtmad_1_dft[d3_name] = np.append(
                        wtmad_1_dft[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"dft{d3_name}"]),
                    )
                    wtmad_2_ai[d3_name] = np.append(
                        wtmad_2_ai[d3_name],
                        data_subset[name_subset][f"ai{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                    wtmad_2_dft[d3_name] = np.append(
                        wtmad_2_dft[d3_name],
                        data_subset[name_subset][f"dft{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                mean_absolute_deviation_list = np.append(
                    mean_absolute_deviation_list,
                    data_subset[name_subset]["cc"],
                )
            if (
                len(data_subset[name_subset]["ai"])
                == len(data_subset[name_subset]["name"])
                and len(data_subset[name_subset]["name"]) > 0
            ):
                processed.append(1)
            else:
                processed.append(0)
        for d3_name in ["", "_d3bj", "_d3zero"]:
            mean_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(subset_ai[d3_name])
            )
            mean_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(subset_dft[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(wtmad_1_ai[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(wtmad_1_dft[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.sum(wtmad_2_ai[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.sum(wtmad_2_dft[d3_name])
            )
        mean_subset.loc[name_set, (data_path_name, "Processed")] = (
            f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_1_subset.loc[name_set, (data_path_name, "Processed")] = (
            f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_2_subset.loc[name_set, (data_path_name, "Processed")] = (
            f"{sum(processed)} / " f"{len(processed)}"
        )

    mean_absolute_deviation = np.mean(mean_absolute_deviation_list) / len(mean_absolute_deviation_list)
    for name_set in full_subset_dict.keys():
        for d3_name in ["", "_d3bj", "_d3zero"]:
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[
                    name_set, (data_path_name, f"DFT{d3_name.upper()}")
                ]
            )

print("Summary")
print("MAE")
display(mean_subset)
print("wtmad_1")
display(wtmad_1_subset)
print("wtmad_2")
display(wtmad_2_subset)
print("Summary of Subset")
print("MAE")
display(df_summary_subset)

# save summary to csv with date
df_summary_subset.to_csv(f"../validate/summary_subset_{date}.csv")
# save summary to excel with date
df_summary_subset.to_excel(f"../validate/summary_subset_{date}.xlsx")

cc-pVDZ
Summary
MAE


data_path   1424849                                                       \
Disp type        AI        DFT   AI_D3BJ   DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1       2.002477  13.710857  2.341162  14.259701  1.840461  13.713797   
sub2       5.510109   6.612164   9.25764   9.130035  6.192243   6.757695   
sub3       2.617318   6.261376  3.548881   7.355579  3.228945   6.934991   
sub4        1.93524   3.272197  2.924356    5.03074  2.900361   4.988172   
sub5       1.626727   1.321288   1.52729   0.928323  1.545785   0.872793   

data_path             1477959                                            \
Disp type Processed        AI        DFT   AI_D3BJ   DFT_D3BJ AI_D3ZERO   
sub1        18 / 18  4.024911  13.710857  4.394827  14.259701  3.888615   
sub2          7 / 9  6.165144   6.612164  9.243087   9.130035   6.56423   
sub3          7 / 7  3.280836   6.261376   4.21537   7.355579   3.92612   
sub4        11 / 12  1.861269   3.272197  2.940841    5.03074  2.901616   
sub5          8 / 9  2.264135   1.321288   1.52166   0.928323  1.487843   

data_path                       
Disp type DFT_D3ZERO Processed  
sub1       13.713797   18 / 18  
sub2        6.757695     7 / 9  
sub3        6.934991     7 / 7  
sub4        4.988172   11 / 12  
sub5        0.872793     8 / 9

wtmad_1


data_path    1424849                                                        \
Disp type         AI        DFT    AI_D3BJ  DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
sub1        3.239113   7.420922   2.966668  7.414578   2.651677   7.102094   
sub2       13.142366   12.53052    11.9277  9.968131  11.524608  10.005264   
sub3        4.832263   6.984991   5.854163  8.150912   5.522858   7.521852   
sub4       11.556831  11.029773  13.901938  14.87552  12.876599  13.798614   
sub5       15.335367  11.571826  14.145431  8.092645  14.544885   7.523212   

data_path              1477959                                             \
Disp type Processed         AI        DFT    AI_D3BJ  DFT_D3BJ  AI_D3ZERO   
sub1        18 / 18   3.789626   7.420922   3.443295  7.414578   3.046364   
sub2          7 / 9  11.038943   12.53052   9.121917  9.968131   9.389425   
sub3          7 / 7   5.389792   6.984991   6.327282  8.150912   6.104995   
sub4        11 / 12  11.131577  11.029773  14.344735  14.87552  13.234994   
sub5          8 / 9  20.741529  11.571826    13.6042  8.092645  13.667041   

data_path                       
Disp type DFT_D3ZERO Processed  
sub1        7.102094   18 / 18  
sub2       10.005264     7 / 9  
sub3        7.521852     7 / 7  
sub4       13.798614   11 / 12  
sub5        7.523212     8 / 9

wtmad_2


data_path   1424849                                                     \
Disp type        AI       DFT   AI_D3BJ  DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1       1.438234  4.040696  1.379994  4.111446  1.263846   3.989769   
sub2       3.339073  3.635331  2.970771  2.376009  2.735087   2.545658   
sub3       1.303233  2.611064   1.66299  3.040013  1.549133   2.873647   
sub4       4.683102  4.159493  6.811273  6.856534  6.578153    6.62018   
sub5       6.331811   4.66236   6.30221  3.511844  6.387196    3.19725   

data_path             1477959                                          \
Disp type Processed        AI       DFT   AI_D3BJ  DFT_D3BJ AI_D3ZERO   
sub1        18 / 18  1.921023  4.040696  1.839362  4.111446   1.68376   
sub2          7 / 9  3.675662  3.635331  2.522968  2.376009  2.765968   
sub3          7 / 7   1.54398  2.611064  1.925105  3.040013  1.817199   
sub4        11 / 12  4.306403  4.159493  6.554155  6.856534  6.318974   
sub5          8 / 9  8.384364   4.66236  5.942533  3.511844   5.89853   

data_path                       
Disp type DFT_D3ZERO Processed  
sub1        3.989769   18 / 18  
sub2        2.545658     7 / 9  
sub3        2.873647     7 / 7  
sub4         6.62018   11 / 12  
sub5         3.19725     8 / 9

Summary of Subset
MAE


data_path    1424849                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
W4_11       1.028249  29.506863   2.441523  31.211753   1.248829  29.861055   
G21EA       0.816732   9.754522   0.812414   9.751745   0.823677   9.757546   
G21IP       0.694714   8.953334   0.702841   8.946699   0.685192     8.9519   
DIPCS10     2.042735  12.310852   2.034436  12.334099   2.049052  12.264655   
PA26        3.009087   2.200766   2.998221   1.900152   2.951891    2.00824   
SIE4x4      3.430805  21.908516   3.716252  22.337814   3.729847  22.339928   
ALKBDE10    0.959132  18.125246   1.307356  18.838948    0.94979  18.135588   
YBDE18      5.939045   8.145393   3.828518   7.265227   4.255001   7.472761   
AL2X6       6.140013   5.659145   2.086328   1.332003    2.63339   1.829895   
HEAVYSB11   2.024005   5.391476   3.570429   8.010474   1.845095   5.984154   
NBPRC       3.058861    2.23255   2.325813   2.544088   2.192134   2.274234   
ALK8        4.403777   4.400063   3.075723   3.174803   2.009981   2.559066   
RC21        2.456062   4.820926   2.009509   6.671006   1.816792     6.0297   
G2RC        1.887093   5.917203    2.60955   6.933515   2.145182   6.417298   
BH76RC       0.85212   3.484561   1.035153   3.539828    0.96494   3.536982   
FH51        2.464205   3.703657   2.323411    3.39478   2.136612   3.256908   
TAUT15      1.702465    2.16453   1.631979   2.175485   1.578771   2.145937   
DC13        6.170057  13.090861   7.562068  12.474047   5.746533  11.475687   
MB16_43     13.39941  15.461604  35.416173  36.862998  21.115024  23.165472   
DARC        7.823359   10.75415   2.327124    3.43411   3.060175    5.57797   
RSE43        3.11738     3.1576   2.896011   2.916276   2.755026    2.74911   
BSR36        4.98276    8.40118   3.343296   1.213694   2.368881   3.079106   
CDIE20      1.450234   1.599507   1.483496   1.336544   1.479584   1.347484   
ISO34       2.020035   2.001743   1.659329   1.577125   1.708557   1.647042   
ISOL24             0          0          0          0          0          0   
C60ISO             0          0          0          0          0          0   
PArel       3.015433   1.743933   2.882746   1.733749   2.907704   1.645025   
BH76        2.803583   9.107946   3.232992     9.8892   3.093517   9.676053   
BHPERI      2.657111   3.268992    6.88257   7.804724   5.491745     6.4139   
BHDIV10     4.741552   6.277706   5.390696   7.448955    4.85035   6.535109   
INV24       3.393792   2.697333   3.863218   2.346995   3.631489   2.228547   
BHROT27     1.713103   0.853379   1.734055   0.858852   1.763036    0.78352   
PX13        1.059156  11.042023   1.291815   11.82601   1.212867  11.219857   
WCPT18      2.039614   7.967151   2.977303   9.151986   2.749679   8.744304   
RG18         0.30936   0.230018   0.765676   0.655714   0.819514   0.709552   
ADIM6       3.598266   3.059029   2.516398   2.090256   2.777618   2.390824   
S22         2.728845   2.291252   3.309309   2.402928   3.196728   2.291758   
S66         2.326488   1.894709    1.79261   2.058561   1.834235   2.101976   
HEAVY28            0          0          0          0          0          0   
WATER27      3.02651  16.299413   8.262183  25.491048   9.177355  26.402884   
CARBHB12    0.484944   1.632408   1.161335   2.973103   1.030009   2.841777   
PNICO23     0.840943   0.776429   2.313413   2.604046    1.44661   1.726817   
HAL59       2.035403   1.649786   2.945249   2.629912   2.562614   2.247276   
AHB21       1.666768   2.453821   2.321702   3.464384   2.212706   3.294519   
CHB6        1.766454    1.80532   1.334639   3.038126   1.432927   2.309161   
IL16        1.467889   1.021067   3.988457   4.339132   4.059287   4.409963   
IDISP      13.356085  13.125475   5.787398   4.423061   6.653317   5.245746   
ICONF       0.946016   0.413804    0.81078   0.464652   0.905493   0.493337   
ACONF       3.016487   0.546621   2.340276   0.430

In [4]:
# import numpy as np
np.max(mean_absolute_deviation_list), np.sum(mean_absolute_deviation_list), np.mean(
    mean_absolute_deviation_list
)

(np.float64(1207.482739432191),
 np.float64(98414.09024557225),
 np.float64(71.1084467092285))

In [5]:
mole_name

'BH76RC-NH'

In [9]:
e1 = -4.6025270679642568e02  # orca fc DLPNO-CCSD
e2 = -4.6025221729046751e02  # orca fc CCSD
e3 = -4.6025767731174767e02 # orca nfc CCSD
e4 = -4.6025796979352134e02 # orca nfc DLPNO-CCSD
e = -460.257677434825  # pyscf CCSD
print((np.array([e1, e2, e3, e4]) - e) * 627.5094733748099)
print((np.array([e1, e2, e3, e4]) - e) / e)

[ 3.11912268e+00  3.42629231e+00  7.72321803e-05 -1.83457852e-01]
[-1.07996860e-05 -1.18632336e-05 -2.67409583e-10  6.35206561e-07]


In [14]:
e = -460.2602125077868 # pyscf CCSD(t)
e1 = -4.6026021233042377e02 # orca nfc CCSD(t)
e2 = -4.6026021233042360e02 # orca old nfc CCSD(t)
e3 = -4.6026042971888393e02  # orca nfc DLPNO-CCSD(t)
e4 = -4.6025500524617820e02
print((np.array([e1, e2, e3, e4]) - e) * 627.5094733748099)
print((np.array([e1, e2, e3, e4]) - e) / e)

[ 1.11296967e-04  1.11297074e-04 -1.36302021e-01  3.26760599e+00]
[-3.85353766e-10 -3.85354136e-10  4.71931076e-07 -1.13137340e-05]


In [ ]:
np.array([[-4.366423914512e-02], [0.000000000000e00], [-5.569827800646e-01]])
np.array([[-4.366423911029e-02], [0.000000000000e00], [-5.569827799431e-01]])

[[-4.366423908914e-02], [0.000000000000e00], [-5.569827798789e-01]]

array([[-3.48300000e-11],
       [ 0.00000000e+00],
       [-1.21500032e-10]])